# Import participants + appariement automatique des CNI (recto/verso) → PDF individuels

**Flux :**
1. ⚙️ Configuration
2. 📦 Dépendances OCR *(une seule fois)*
3. 🔑 Authentification
4. 🔍 Extraction OCR d'un dossier de photos CNI (n° CNI + recto/verso auto-détectés)
5. 🧩 Appariement recto↔verso + matching avec la liste Excel → **CSV de contrôle**
6. ✍️ *(tu relis / corriges / valides le CSV de contrôle)*
7. 🏢 Création de l'activité
8. 👥 Import des participants avec leurs photos CNI jointes
9. 📑 Génération des PDF individuels

> **Clé d'appariement = le numéro de CNI.** Il est présent sur le **recto** (champ `n°`),
> dans la **MRZ** du verso, et dans la colonne `N° CNI` de l'**Excel**.
>
> La MRZ du verso est lue **directement par Tesseract** (police monospace, faite pour l'OCR)
> puis parsée en pur Python — pas de dépendance à `passporteye`/`scipy`.
>
> ℹ️ Comme l'Excel fusionne nom et prénoms en une colonne, le **découpage nom/prénom**
> vient en priorité de la MRZ (qui les sépare) ; à défaut, découpage auto (1ᵉʳ mot = nom).
>
> ⚠️ Le CSV de contrôle est un **garde-fou** : relis-le avant l'import.

## ⚙️ 1. Configuration

In [ ]:
import os

# ── À renseigner ────────────────────────────────────────────────────
# Les identifiants viennent de l'environnement : ce notebook est suivi
# par git, un mot de passe écrit ici finirait dans l'historique du dépôt.
#   export PRESENCE_USER=... PRESENCE_PASSWORD=...   (puis relancer jupyter)
API_URL      = os.environ.get("PRESENCE_API_URL", "http://localhost:8000")
USERNAME     = os.environ["PRESENCE_USER"]
PASSWORD     = os.environ["PRESENCE_PASSWORD"]

EXCEL_PATH   = r"liste_duekoue.xlsx"
PHOTOS_DIR   = r"photos_duekoue"     # Dossier des photos CNI (recto + verso en vrac)
OUTPUT_DIR   = r"PDFs_Duekoue"       # Dossier de sortie des PDF

CONTROL_CSV  = r"controle_cni_duekoue.csv"  # CSV de contrôle généré par l'OCR

# ⚠️ Mode temporaire : pré-valide aussi les participants dont le recto et/ou le verso
# de la CNI n'a pas été reconnu (photo manquante, illisible...), pour ne bloquer la
# création de personne. Repasse à False pour revenir au comportement strict
# (seul un statut OK — recto ET verso appariés — est auto-validé).
IMPORTER_SANS_CNI_COMPLETE = True

# Chemin de l'exécutable Tesseract (binaire installé séparément — cf. cellule Dépendances).
# Mets None si tesseract est déjà accessible dans le PATH.
TESSERACT_CMD = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
# ─────────────────────────────────────────────────────────────────

import os, re, requests, pandas as pd
from pathlib import Path

PHOTOS_REDRESSEES_DIR = Path(PHOTOS_DIR) / "_redressees"  # copies couleur tête en haut (générées à l'étape 4)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Configuration OK")

## 📦 2. Dépendances OCR (à installer une seule fois)

L'extraction **100 % locale** repose sur **Tesseract** (moteur OCR) + quelques libs Python.

**a) Le binaire Tesseract** (hors Python) :
- Installeur Windows : https://github.com/UB-Mannheim/tesseract/wiki
- Pendant l'installation, **coche le pack de langue français (`fra`)**.
- Reporte le chemin d'installation dans `TESSERACT_CMD` (cellule Configuration).

**b) Les libs Python** — depuis **la racine du dépôt** (pas depuis `backend/`) :
```
uv sync --extra import
```
> Déclarées dans le `pyproject.toml` **de la racine**, sous
> `[project.optional-dependencies].import` : `ipykernel`, `openpyxl`, `pandas`,
> `pillow`, `pytesseract`, `requests`. Léger, sans scipy/scikit-learn.
>
> ⚠️ Cet environnement (`.venv` à la racine) est **distinct de celui du backend**
> (`backend/.venv`, décrit par `backend/pyproject.toml`). Le notebook n'importe
> jamais Django, et le backend n'a besoin ni de pandas ni de l'OCR. Lancer
> `uv sync` depuis `backend/` installerait donc dans le mauvais environnement.

**c) Le noyau Jupyter** : dans VS Code, sélectionne en haut à droite le kernel
**« Python (listePresence .venv) »** — celui de la racine, sinon les paquets ne
sont pas reconnus.

In [ ]:
# Vérifie que la stack OCR est disponible et que Tesseract répond.
import importlib.util
import openpyxl
manquants = [m for m in ("pytesseract", "PIL", "openpyxl")
             if importlib.util.find_spec(m) is None]
if manquants:
    raise ImportError(
        f"Libs Python manquantes : {manquants}\n"
        "→ Lance :  uv sync --extra import  (et sélectionne le kernel du .venv)"
    )

import pytesseract
if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

try:
    version = pytesseract.get_tesseract_version()
except Exception as e:
    raise RuntimeError(
        "Binaire Tesseract introuvable. Installe-le puis corrige TESSERACT_CMD.\n"
        f"Détail : {e}"
    )

langues = pytesseract.get_languages(config="")
if "fra" not in langues:
    print("⚠️  Pack de langue 'fra' absent — l'OCR du recto sera moins précis.\n"
          "    Réinstalle Tesseract en cochant le français, ou utilise 'eng'.")
print(f"✅ Tesseract {version} prêt — langues dispo : {', '.join(sorted(langues)[:10])}…")

## 🔑 3. Authentification

In [ ]:
r = requests.post(f"{API_URL}/api/auth/login", json={"username": USERNAME, "password": PASSWORD})
r.raise_for_status()
TOKEN = r.json()["access"]
HEADERS = {"Authorization": f"Bearer {TOKEN}"}
print("✅ Authentifié")

## 🔍 4. Extraction OCR des CNI

Pour chaque image :
- d'abord, l'image est **redressée tête en haut** (détection d'orientation Tesseract — OSD ;
  repli sur l'heuristique portrait→paysage si l'OSD n'a pas assez de texte pour conclure) ;
- on tente de lire une **MRZ** (Tesseract avec whitelist `A-Z 0-9 <`) → si trouvée, c'est un **verso** :
  on en tire le n° CNI et le nom/prénoms (séparés) ;
- sinon → c'est un **recto** : OCR classique et lecture du champ `n°`.

Les fonctions font l'extraction ; la cellule suivante les exécute sur tout le dossier.

In [ ]:
# ── Fonctions d'extraction & normalisation (Tesseract pur) ────────────────
from PIL import Image, ImageOps

MRZ_WL = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789<"   # caractères autorisés dans une MRZ


def _normaliser_cni(brut: str | None) -> str | None:
    """n° CNI canonique 'CI' + 9 chiffres, ou None.

    Tolère les confusions de saisie/OCR : O→0 et le préfixe 'C1' pour 'CI'.
    Dans une MRZ, le n° déborde (ICAO TD1) : 'CI0123456' + '<' + '194' →
    après retrait des '<' : 'CI0123456789' ; on garde 'CI' + 9 chiffres =
    'CI012345619' (la clé de contrôle finale est naturellement écartée).
    """
    if not brut:
        return None
    s = brut.upper().replace("O", "0").replace("<", "").replace(" ", "")
    m = re.search(r"C[I1](\d{9})", s)   # CI ou C1 (mauvaise saisie)
    return f"CI{m.group(1)}" if m else None


def _normaliser_tel(brut: str | None) -> str:
    """Ramène un numéro à 10 chiffres locaux (Excel a perdu le 0 initial → 9 chiffres)."""
    d = re.sub(r"\D", "", brut or "")
    if d.startswith("225") and len(d) == 13:
        d = d[3:]
    if len(d) == 9:
        d = "0" + d
    return d


def _split_nom_prenom(complet: str | None) -> tuple[str, str, str]:
    """Découpe un nom complet Excel faute de MRZ : 1ᵉʳ mot = nom, reste = prénoms."""
    toks = (complet or "").split()
    if not toks:
        return "", "", "excel(vide)"
    if len(toks) == 1:
        return toks[0], "", "excel(1mot)"
    return toks[0], " ".join(toks[1:]), "excel(split)"


def _lignes_mrz(texte: str) -> list[str]:
    """Isole les lignes ressemblant à une MRZ : longues et composées de A-Z/0-9/<."""
    lignes = []
    for raw in texte.splitlines():
        s = re.sub(r"[^A-Z0-9<]", "", raw.upper())
        if len(s) >= 20 and "<" in s:
            lignes.append(s)
    return lignes


def _noms_depuis_mrz(lignes: list[str]) -> tuple[str, str]:
    """Extrait (nom, prénoms) de la ligne des noms d'une MRZ (SURNAME<<GIVEN<NAMES)."""
    for s in lignes:
        if s.startswith("ID"):          # ligne 1 (document), pas les noms
            continue
        if "<<" in s and re.search(r"[A-Z]{2}", s):
            surname, _, given = s.partition("<<")
            nom = re.sub(r"<+", " ", surname).strip()
            prenom = re.sub(r"<+", " ", given).strip()
            if nom:
                return nom, prenom
    return "", ""


def _detecter_rotation(img) -> int:
    """Angle (0/90/180/270) à appliquer pour remettre le texte à l'endroit, via l'OSD Tesseract.

    Le rapport largeur/hauteur seul ne dit pas si un portrait (ou un paysage) a été pris
    à l'envers ; l'OSD lit le sens réel du texte et lève cette ambiguïté. Repli sur
    l'heuristique portrait→paysage (90°) si l'OSD ne trouve pas assez de texte pour conclure
    (photo trop bruitée/floue).
    """
    try:
        osd = pytesseract.image_to_osd(img, output_type=pytesseract.Output.DICT)
        angle = int(osd.get("rotate", 0) or 0)
    except Exception:
        angle = 0
    if not angle and img.height > img.width:
        angle = 90
    return angle


def _pivoter(img, angle: int):
    """Applique la rotation détectée (sens horaire), sans rogner (expand=True)."""
    return img.rotate(-angle, expand=True) if angle else img


def _preparer_ocr(img):
    """Réduit (≤ 2000 px de large) une image déjà redressée et la passe en gris contrasté.

    Le redimensionnement accélère l'OCR (~3×) et évite les blocages de Tesseract
    sur les très grandes photos (4080 px), sans nuire à la lisibilité de la CNI.
    """
    if img.width > 2000:
        img = img.resize((2000, round(img.height * 2000 / img.width)))
    return ImageOps.autocontrast(img.convert("L"))


def _ocr(img, psm: int, whitelist: str | None = None, lang: str = "eng") -> str:
    """Lance Tesseract sur une image déjà préparée (psm + whitelist optionnels)."""
    cfg = f"--psm {psm}"
    if whitelist:
        cfg += f" -c tessedit_char_whitelist={whitelist}"
    return pytesseract.image_to_string(img, lang=lang, config=cfg)


def extraire_cni(path: Path, dest_dir: Path | None = None) -> dict:
    """Analyse une image de CNI → {face, numero_cni, nom, prenom, confiance, remarque, rotation}.

    Si dest_dir est fourni, une copie couleur pleine résolution de l'image redressée
    (tête en haut) y est enregistrée sous le même nom de fichier — c'est cette copie
    qui sera utilisée pour l'import et les PDF, pas l'original potentiellement pivoté.
    """
    res = {"fichier": path.name, "face": "?", "numero_cni": None,
           "nom": "", "prenom": "", "confiance": "", "remarque": "", "rotation": 0}
    try:
        img = ImageOps.exif_transpose(Image.open(path))
        angle = _detecter_rotation(img)
        img = _pivoter(img, angle)
        res["rotation"] = angle
        if dest_dir is not None:
            dest_dir.mkdir(parents=True, exist_ok=True)
            img.convert("RGB").save(dest_dir / path.name)
        img_ocr = _preparer_ocr(img)
    except Exception as e:
        res["remarque"] = f"image illisible: {e}"
        return res

    # 1) Tente de lire une MRZ (whitelist MRZ). Si ≥ 2 lignes MRZ → VERSO.
    lignes = _lignes_mrz(_ocr(img_ocr, 6, MRZ_WL))
    if len(lignes) >= 2:
        res["face"] = "verso"
        res["numero_cni"] = _normaliser_cni("".join(lignes))
        res["nom"], res["prenom"] = _noms_depuis_mrz(lignes)
        res["confiance"] = f"mrz:{len(lignes)}L"
        if not res["numero_cni"]:
            res["remarque"] = "verso : n° CNI illisible dans la MRZ"
        return res

    # 2) Sinon → RECTO : on lit le champ n° en essayant plusieurs segmentations.
    res["face"] = "recto"
    lang = "fra" if "fra" in langues else "eng"
    for psm in (6, 3, 4):
        num = _normaliser_cni(_ocr(img_ocr, psm, lang=lang))
        if num:
            res["numero_cni"] = num
            res["confiance"] = f"ocr:psm{psm}"
            return res
    res["confiance"] = "ocr"
    res["remarque"] = "recto : n° CNI non détecté par l'OCR"
    return res


print("✅ Fonctions d'extraction chargées (Tesseract pur)")

In [ ]:
# Parcourt le dossier et extrait chaque image (recto ou verso).
photos = sorted(
    p for p in Path(PHOTOS_DIR).iterdir()
    if p.suffix.lower() in (".jpg", ".jpeg", ".png")
)
print(f"📁 {len(photos)} image(s) à analyser dans {PHOTOS_DIR}\n")

extractions = []
for i, p in enumerate(photos, 1):
    r = extraire_cni(p, dest_dir=PHOTOS_REDRESSEES_DIR)
    extractions.append(r)
    flag = "✅" if r["numero_cni"] else "⚠️ "
    rot = f"↻{r['rotation']}°" if r["rotation"] else ""
    print(f"  {flag} [{i}/{len(photos)}] {p.name[:32]:32s} "
          f"{r['face']:6s} {r['numero_cni'] or '— illisible —':12s} {rot:5s} {r['remarque']}")

df_ocr = pd.DataFrame(extractions)
lus = int(df_ocr["numero_cni"].notna().sum()) if not df_ocr.empty else 0
print(f"\n🔍 Extraction terminée : {lus}/{len(df_ocr)} numéro(s) lu(s)")
print(f"🖼️  Copies redressées (tête en haut) enregistrées dans : {PHOTOS_REDRESSEES_DIR}")

## 🧩 5. Appariement recto↔verso + matching Excel → CSV de contrôle

On regroupe les photos par numéro de CNI (recto + verso = une paire), puis on croise
avec la liste Excel (clé = `N° CNI`, normalisée). Le résultat va dans `CONTROL_CSV`.

- `nom` / `prenom` viennent de la **MRZ** quand un verso est lisible (`source_identite = mrz`),
  sinon du découpage automatique du nom Excel (`source_identite = excel(split)`).
- `nom_complet_excel` est conservé en regard pour vérification.
- `telephone_wave` est re-préfixé du `0` manquant ; les n° CNI mal saisis sont corrigés.

In [ ]:
# ── Lecture de l'Excel (tout en texte pour préserver téléphones / CNI) ─────────
df_excel = pd.read_excel(EXCEL_PATH, dtype=str).fillna("")
df_excel.columns = df_excel.columns.str.strip()
col_map = {
    "Nom & Prénom (Comme sur la CNI)": "nom_complet",
    "STRUCTURE": "structure",
    "Fonction": "fonction",
    "Contact(wave)": "telephone_wave",
    "EMAIL1": "email",
    "N° CNI": "numero_cni",
}
manquantes = [c for c in col_map if c not in df_excel.columns]
assert not manquantes, (
    f"Colonnes Excel introuvables : {manquantes}\n"
    f"Colonnes présentes : {list(df_excel.columns)}"
)
df_excel = df_excel.rename(columns=col_map)[list(col_map.values())].copy()
df_excel["cni_norm"] = df_excel["numero_cni"].map(_normaliser_cni)

# ── Regroupe les photos lues par n° CNI (+ noms issus de la MRZ) ─────────────
paires: dict[str, dict] = {}
illisibles = []
for r in extractions:
    cni = r["numero_cni"]
    if not cni:
        illisibles.append(r)
        continue
    g = paires.setdefault(cni, {"recto": None, "verso": None, "conf": [],
                                "nom_mrz": "", "prenom_mrz": ""})
    g[r["face"]] = r["fichier"]
    g["conf"].append(r["confiance"])
    if r["face"] == "verso" and r["nom"]:
        g["nom_mrz"], g["prenom_mrz"] = r["nom"], r["prenom"]

# ── Construit le CSV de contrôle depuis l'Excel (la liste fait foi) ───────────
lignes = []
for _, row in df_excel.iterrows():
    cni = row["cni_norm"]
    g = paires.get(cni, {})
    recto, verso = g.get("recto"), g.get("verso")

    # Identité : MRZ prioritaire (nom/prénoms séparés), sinon découpage du nom Excel.
    if g.get("nom_mrz"):
        nom, prenom, source = g["nom_mrz"], g["prenom_mrz"], "mrz"
    else:
        nom, prenom, source = _split_nom_prenom(row["nom_complet"])

    if cni is None:
        statut = "CNI_EXCEL_INVALIDE"      # n° absent / illisible dans l'Excel
    elif recto and verso:
        statut = "OK"
    elif recto or verso:
        statut = "PHOTO_INCOMPLETE"        # une seule face trouvée
    else:
        statut = "SANS_PHOTO"              # aucune photo pour cette personne

    # SANS_PHOTO n'est exclu du CSV qu'en mode strict — en mode temporaire, la
    # personne doit quand même pouvoir être créée (juste sans photo de CNI).
    if statut == "SANS_PHOTO" and not IMPORTER_SANS_CNI_COMPLETE:
        continue

    lignes.append({
        "valider": "oui" if (statut == "OK" or IMPORTER_SANS_CNI_COMPLETE) else "",
        "statut": statut,
        "numero_cni": cni or row["numero_cni"],
        "nom": nom, "prenom": prenom, "source_identite": source,
        "nom_complet_excel": row["nom_complet"].strip(),
        "structure": row["structure"].strip(), "fonction": row["fonction"].strip(),
        "telephone_wave": _normaliser_tel(row["telephone_wave"]),
        "email": row["email"].strip(),
        "photo_recto": recto or "", "photo_verso": verso or "",
        "confiance": ";".join(g.get("conf", [])),
    })

df_controle = pd.DataFrame(lignes)
df_controle.to_csv(CONTROL_CSV, index=False, encoding="utf-8-sig")

# ── Rapport ────────────────────────────────────────────────────────
cni_excel = set(df_excel["cni_norm"].dropna())
orphelines = set(paires) - cni_excel   # CNI photographiées mais absentes de l'Excel
print(f"✅ CSV de contrôle écrit : {CONTROL_CSV}\n")

if IMPORTER_SANS_CNI_COMPLETE:
    incomplets = int((df_controle["statut"] != "OK").sum())
    print(f"⚠️  Mode temporaire actif (IMPORTER_SANS_CNI_COMPLETE=True) : {incomplets} "
          f"participant(s) sans CNI complète sont quand même pré-validés (valider=oui).\n"
          f"    Mets 'valider' à vide dans le CSV pour exclure ceux que tu ne veux pas "
          f"importer sans photo.\n")

print("Répartition des statuts :")
print(df_controle["statut"].value_counts().to_string(), "\n")
print("Source de l'identité (nom/prénom) :")
print(df_controle["source_identite"].value_counts().to_string(), "\n")

print(f"📸 Photos illisibles (n° non détecté) : {len(illisibles)}")
for r in illisibles:
    print(f"    - {r['fichier']} ({r['face']}) — {r['remarque']}")

print(f"\n🚫 CNI photographiées absentes de l'Excel : {len(orphelines)}")
for cni in sorted(orphelines):
    g = paires[cni]
    print(f"    - {cni}  recto={g['recto']}  verso={g['verso']}")

df_controle.head(10)

## ✍️ 6. RELIRE ET VALIDER le CSV de contrôle

> **Ouvre** `controle_cni_duekoue.csv` (Excel ou éditeur).
>
> ⚠️ **Mode temporaire actif** (`IMPORTER_SANS_CNI_COMPLETE = True`) : tous les participants
> de l'Excel sont pré-validés (`valider=oui`), **même sans CNI complète** (recto et/ou verso
> non reconnu). Les colonnes `photo_recto`/`photo_verso` restent vides pour ces lignes — ils
> seront créés sans photo de CNI, complétable plus tard depuis l'application.
>
> Vérifie surtout :
> - **`source_identite`** : les lignes `excel(split)` (pas de verso lisible) ont un découpage
>   automatique `nom`/`prenom` — compare-les à `nom_complet_excel` et corrige si besoin.
> - `photo_recto` / `photo_verso` correspondent **bien** à la bonne personne.
> - Colonne **`valider`** = `oui` → importée ; **vide** → ignorée. Vide une ligne si tu
>   préfères exclure un participant sans CNI complète plutôt que de le créer sans photo.
> - `PHOTO_INCOMPLETE` / `SANS_PHOTO` : complète la photo manquante à la main si tu l'as ;
>   sinon laisse tel quel, la personne sera importée sans elle.
> - Traite les **illisibles** et **CNI orphelines** du rapport ci-dessus.
>
> **Sauvegarde le CSV**, puis exécute la suite.

## 🏢 7. Création de l'activité

In [ ]:
# ── Renseigne les informations de l'activité ────────────────────────────
ACTIVITE = {
    "nom":         "Formation Duékoué SNGFA",
    "description": "",
    "date_debut":  "2026-06-01T08:00:00",   # Format ISO 8601
    "date_fin":    "2026-06-05T16:30:00",
    "ville":       "Duékoué",
    "lieu":        "",
}
# ─────────────────────────────────────────────────────────────────

r = requests.post(f"{API_URL}/api/activites/", json=ACTIVITE, headers=HEADERS)
r.raise_for_status()
activite = r.json()
ACTIVITE_ID = activite["id"]
print(f"✅ Activité créée : {activite['nom']}")
print(f"   ID : {ACTIVITE_ID}")

## 👥 8. Import des participants avec photos

In [ ]:
df_match = pd.read_csv(CONTROL_CSV, encoding="utf-8-sig", dtype=str).fillna("")
a_importer = df_match[df_match["valider"].str.strip().str.lower() == "oui"].reset_index(drop=True)
print(f"👥 {len(a_importer)}/{len(df_match)} ligne(s) validée(s) à importer\n")

resultats = []
for i, row in a_importer.iterrows():
    nom_complet = f"{row['prenom']} {row['nom']}".strip()

    # Données texte
    data = {
        "nom":            row["nom"].strip(),
        "prenom":         row["prenom"].strip(),
        "structure":      row["structure"].strip(),
        "fonction":       row["fonction"].strip(),
        "telephone_wave": row["telephone_wave"].strip(),
        "email":          row["email"].strip(),
        "numero_cni":     row["numero_cni"].strip(),
    }

    # Photos (optionnelles) — on garde les handles pour les fermer après l'envoi.
    files, ouverts = {}, []
    for cote in ("recto", "verso"):
        fname = row.get(f"photo_{cote}", "").strip()
        if fname:
            fpath = Path(PHOTOS_REDRESSEES_DIR) / fname
            if not fpath.exists():
                fpath = Path(PHOTOS_DIR) / fname   # repli : photo ajoutée à la main après l'étape 4
            if fpath.exists():
                fobj = open(fpath, "rb")
                ouverts.append(fobj)
                files[f"photo_cni_{cote}"] = (fname, fobj, "image/jpeg")
            else:
                print(f"  ⚠️  [{nom_complet}] fichier {cote} introuvable : {fname}")

    r = requests.post(
        f"{API_URL}/api/activites/{ACTIVITE_ID}/participants",
        data=data,
        files=files or None,
        headers=HEADERS,
    )
    for fobj in ouverts:
        fobj.close()

    if r.status_code == 201:
        resultats.append({"nom_complet": nom_complet, "id": r.json()["id"], "statut": "OK"})
        print(f"  ✅ [{i+1}/{len(a_importer)}] {nom_complet}")
    elif r.status_code == 409:
        resultats.append({"nom_complet": nom_complet, "id": None, "statut": "DOUBLON CNI"})
        print(f"  ⚠️  [{i+1}/{len(a_importer)}] {nom_complet} — doublon CNI ignoré")
    else:
        resultats.append({"nom_complet": nom_complet, "id": None, "statut": f"ERREUR {r.status_code}"})
        print(f"  ❌ [{i+1}/{len(a_importer)}] {nom_complet} — {r.status_code}: {r.text[:120]}")

df_resultats = pd.DataFrame(resultats)
ok = int((df_resultats["statut"] == "OK").sum()) if not df_resultats.empty else 0
print(f"\n🎉 Import terminé : {ok}/{len(a_importer)} participant(s) importé(s) avec succès")

## 📑 9. Génération des PDF individuels

In [ ]:
importes = df_resultats[df_resultats["statut"] == "OK"]
print(f"Génération de {len(importes)} PDF...")

for _, row in importes.iterrows():
    r = requests.get(   
        f"{API_URL}/api/exports/participants/{row['id']}/pdf",
        headers=HEADERS,
    )
    if r.status_code == 200:
        # Nom de fichier : NOM_PRENOM_cni.pdf
        safe_name = row["nom_complet"].replace(" ", "_").replace("/", "-")
        pdf_path = Path(OUTPUT_DIR) / f"{safe_name}_cni.pdf"
        pdf_path.write_bytes(r.content)
        print(f"  ✅ {pdf_path.name}")
    else:
        print(f"  ❌ {row['nom_complet']} — {r.status_code}")

print(f"\n🎉 PDF sauvegardés dans : {OUTPUT_DIR}")